# Project 05: House Price Prediction\n## Notebook 02: Multi-Model Sync

In [1]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import StackingRegressor
import pickle
import os

# Load cleaned data
train = pd.read_csv('../data/processed/train_cleaned.csv')

# Engineering
train['TotalSF'] = train['TotalBsmtSF'] + train['1stFlrSF'] + train['2ndFlrSF']
train['Total_Bathrooms'] = (train['FullBath'] + (0.5 * train['HalfBath']) +
                             train['BsmtFullBath'] + (0.5 * train['BsmtHalfBath']))
train['Qual_Area_Interact'] = train['OverallQual'] * train['GrLivArea']

X = train.drop(['Id', 'SalePrice', 'SalePrice_Log'], axis=1)
y = train['SalePrice_Log']

# Encoding
qual_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}
qual_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond']
for col in qual_cols:
    if col in X.columns:
        X[col] = X[col].map(qual_map).fillna(0)

X = pd.get_dummies(X)
print(f"Final Feature Space: {X.shape}")


Final Feature Space: (1460, 258)


In [2]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Individual Models
ridge = Ridge(alpha=10)
ridge.fit(X_train_scaled, y_train)

xgb = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=3, random_state=42)
xgb.fit(X_train_scaled, y_train)

lgbm = LGBMRegressor(n_estimators=1000, learning_rate=0.05, max_depth=3, random_state=42, verbose=-1)
lgbm.fit(X_train_scaled, y_train)

# Stacking
estimators = [('ridge', ridge), ('xgb', xgb), ('lgbm', lgbm)]
stack = StackingRegressor(estimators=estimators, final_estimator=Ridge(alpha=1.0))
stack.fit(X_train_scaled, y_train)

# Save All Models for UI Selection
os.makedirs('../models', exist_ok=True)
with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open('../models/stack_model.pkl', 'wb') as f:
    pickle.dump(stack, f)
with open('../models/ridge_model.pkl', 'wb') as f:
    pickle.dump(ridge, f)
with open('../models/xgb_model.pkl', 'wb') as f:
    pickle.dump(xgb, f)
with open('../models/feature_columns.pkl', 'wb') as f:
    pickle.dump(X.columns.tolist(), f)

print("All models synchronized for UI deployment.")


/home/ashutosh/miniconda3/envs/ml-env/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/ashutosh/miniconda3/envs/ml-env/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


/home/ashutosh/miniconda3/envs/ml-env/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/ashutosh/miniconda3/envs/ml-env/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


All models synchronized for UI deployment.


/home/ashutosh/miniconda3/envs/ml-env/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
